# Training Analysis and Visualization

This notebook demonstrates how to visualize ECG samples and analyze the training metrics produced by the training scripts.

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

## Visualizing raw ECG data
Specify the path to one of your raw ECG `.csv` files. Each file should contain at least the columns `Time [s]`, `Signal [mV]` and `Seizure [bool]`.

In [ ]:
csv_path = Path('../data/raw_ecg/example.csv')  # change to your file

if csv_path.exists():
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(df['Time [s]'], df['Signal [mV]'], label='ECG')
    if 'Seizure [bool]' in df.columns:
        ax.fill_between(df['Time [s]'], df['Signal [mV]'].min(), df['Signal [mV]'].max(),
                         where=df['Seizure [bool]'].astype(bool), color='red', alpha=0.2, label='Seizure')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Amplitude [mV]')
    ax.legend()
    plt.show()
else:
    print(f'File {csv_path} not found. Please update the path above.')

## Loading training metrics
After running `train_hierarchical.py` or `train_hybrid_pipeline.py`, a `metrics.csv` file is stored in the run directory.

In [ ]:
metrics_path = Path('../runs/example_run/metrics.csv')  # update with your run

metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
metrics.head()

## Convergence curves

In [ ]:
if not metrics.empty:
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(metrics['epoch'], metrics['train_loss'], label='train loss')
    ax.plot(metrics['epoch'], metrics['val_loss'], label='val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.show()
else:
    print('Metrics dataframe is empty. Make sure metrics_path points to a valid CSV.')

In [ ]:
if not metrics.empty:
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(metrics['epoch'], metrics['auroc'], label='AUROC')
    ax.plot(metrics['epoch'], metrics['auprc'], label='AUPRC')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.legend()
    plt.show()

## Confusion matrix for the test set

In [ ]:
import numpy as np

if not metrics.empty and 'TEST' in metrics['epoch'].values:
    test_row = metrics[metrics['epoch'] == 'TEST'].iloc[0]
    cm = np.array([[test_row['tn'], test_row['fp']],
                   [test_row['fn'], test_row['tp']]])
    fig, ax = plt.subplots()
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Negative','Positive'])
    ax.set_yticklabels(['Negative','Positive'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title('Confusion Matrix')
    plt.show()
else:
    print('Test metrics not found in metrics.csv.')